In [1]:
#Creating DataFrame of Data

import pandas as pd

df=pd.read_csv("file_path")

df.info()

df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'file_path'

In [ ]:
df.describe(include="all")

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
count,3900.000000,3900.000000,3900,3900,3900,3900.000000,3900,3900,3900,3900,3863.000000,3900,3900,3900,3900,3900.000000,3900,3900
unique,NaN,NaN,2,25,4,NaN,50,4,25,4,NaN,2,6,2,2,NaN,6,7
top,NaN,NaN,Male,Blouse,Clothing,NaN,Montana,M,Olive,Spring,NaN,No,Free Shipping,No,No,NaN,PayPal,Every 3 Months
freq,NaN,NaN,2652,171,1737,NaN,96,1755,177,999,NaN,2847,675,2223,2223,NaN,677,584
mean,1950.500000,44.068462,NaN,NaN,NaN,59.764359,NaN,NaN,NaN,NaN,3.750065,NaN,NaN,NaN,NaN,25.351538,NaN,NaN
std,1125.977353,15.207589,NaN,NaN,NaN,23.685392,NaN,NaN,NaN,NaN,0.716983,NaN,NaN,NaN,NaN,14.447125,NaN,NaN
min,1.000000,18.000000,NaN,NaN,NaN,20.000000,NaN,NaN,NaN,NaN,2.500000,NaN,NaN,NaN,NaN,1.000000,NaN,NaN
25%,975.750000,31.000000,NaN,NaN,NaN,39.000000,NaN,NaN,NaN,NaN,3.100000,NaN,NaN,NaN,NaN,13.000000,NaN,NaN
50%,1950.500000,44.000000,NaN,NaN,NaN,60.000000,NaN,NaN,NaN,NaN,3.800000,NaN,NaN,NaN,NaN,25.000000,NaN,NaN
75%,2925.250000,57.000000,NaN,NaN,NaN,81.000000,NaN,NaN,NaN,NaN,4.400000,NaN,NaN,NaN,NaN,38.000000,NaN,NaN


In [ ]:
#Checking Null Values

df.isna().sum()

Customer ID                0
Age                        0
Gender                     0
Item Purchased             0
Category                   0
Purchase Amount (USD)      0
Location                   0
Size                       0
Color                      0
Season                     0
Review Rating             37
Subscription Status        0
Shipping Type              0
Discount Applied           0
Promo Code Used            0
Previous Purchases         0
Payment Method             0
Frequency of Purchases     0
dtype: int64

In [ ]:
#Fixing Null Values

df['Review Rating'].value_counts()

# df.fillna({'Review Rating':df['Review Rating'].median()},inplace=True)

df['Review Rating'] = df.groupby('Category')['Review Rating'].transform(lambda x: x.fillna(x.median()))

df.isna().sum()

Customer ID               0
Age                       0
Gender                    0
Item Purchased            0
Category                  0
Purchase Amount (USD)     0
Location                  0
Size                      0
Color                     0
Season                    0
Review Rating             0
Subscription Status       0
Shipping Type             0
Discount Applied          0
Promo Code Used           0
Previous Purchases        0
Payment Method            0
Frequency of Purchases    0
dtype: int64

In [ ]:
#Fixing Column Names

df.columns = df.columns.str.lower()

df.columns = df.columns.str.replace(' ','_')

df.rename(columns={'purchase_amount_(usd)':'purchase_amount'},inplace=True)

df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'promo_code_used', 'previous_purchases',
       'payment_method', 'frequency_of_purchases'],
      dtype='str')

In [ ]:
#Creating New Column 'age_group'

labels = ['Young Adult','Adult','Middle Aged','Old']

df['age_group'] = pd.qcut(df['age'],q=4,labels=labels)

df[['age_group','age']]

,age_group,age
0,Middle Aged,55
1,Young Adult,19
2,Middle Aged,50
3,Young Adult,21
4,Middle Aged,45
...,...,...
3895,Adult,40
3896,Middle Aged,52
3897,Middle Aged,46
3898,Adult,44


In [ ]:
#Creating New Column 'purchase_frequency'

freq = {
    "Fortnightly":14,
    "Weekly":7,
    "Bi-Weekly":14,
    "Quarterly":90,
    "Monthly":30,
    "Annually":365,
    "Every 3 Months":90
}


df['purchase_frequency'] = df['frequency_of_purchases'].map(freq)

df[['frequency_of_purchases','purchase_frequency']]

,frequency_of_purchases,purchase_frequency
0,Fortnightly,14
1,Fortnightly,14
2,Weekly,7
3,Weekly,7
4,Annually,365
...,...,...
3895,Weekly,7
3896,Bi-Weekly,14
3897,Quarterly,90
3898,Weekly,7


In [ ]:
#Checking Redundant Columns

df['promo_code_used']

df['discount_applied']

(df['discount_applied'] == df['promo_code_used']).all()

df.drop('promo_code_used',axis=1,inplace=True)

df.columns


Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'previous_purchases', 'payment_method',
       'frequency_of_purchases', 'age_group', 'purchase_frequency'],
      dtype='str')

In [ ]:
#Connecting To MySQL

#pip install pymysql sqlalchemy

from sqlalchemy import create_engine

# MySQL connection
username = "root"
password = "your_password"
host = "localhost"
port = "3306"
database = "customer_behavior"

engine = create_engine(f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}")

# Write DataFrame to MySQL
table_name = "customer"   # choose any table name
df.to_sql(table_name, engine, if_exists="replace", index=False)

print("Connected")

Connected!


In [ ]:
#Exporting Cleaned Excel Worksheet

df.to_excel(
    "customer_shopping_cleaned.xlsx",
    index=False,
    engine="openpyxl"
)